## Priti Maurya
## PRN: 260240128033

In [1]:

import os
import sys

os.environ["SPARK_HOME"] = "/home/talentum/spark"
os.environ["PYLIB"] = os.environ["SPARK_HOME"] + "/python/lib"

os.environ["PYSPARK_PYTHON"] = "/usr/bin/python3.6" 
os.environ["PYSPARK_DRIVER_PYTHON"] = "/usr/bin/python3"
sys.path.insert(0, os.environ["PYLIB"] +"/py4j-0.10.7-src.zip")
sys.path.insert(0, os.environ["PYLIB"] +"/pyspark.zip")

os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.3 pyspark-shell'


In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Pyspark Lab exam").enableHiveSupport().getOrCreate()
sc = spark.sparkContext

In [3]:
from pyspark.sql.types import * 
tips_schema = StructType([
    StructField('total_bill', DoubleType(), False),
    StructField('tip', DoubleType(), False),
    StructField('gender', StringType(), False),
    StructField('smoker', StringType(), False),
    StructField('day', StringType(), False),
    StructField('time', StringType(), False),
    StructField('size', IntegerType(), False),
    StructField('price_per_person', DoubleType(), False),
    StructField('Payer_Name', StringType(), False)
])


#### Loading the dataset into sparkdataframe, creating schema, dding header=True , so that first row in csv file is taken as header

In [4]:
#a.) Solution
filepath = 'file:///home/talentum/shared/Q1/tips.csv'
tips_df = spark.read.format('csv').load(filepath, schema = tips_schema,header = True)


tips_df.show(10)


+----------+----+------+------+---+------+----+----------------+------------------+
|total_bill| tip|gender|smoker|day|  time|size|price_per_person|        Payer_Name|
+----------+----+------+------+---+------+----+----------------+------------------+
|     16.99|1.01|Female|    No|Sun|Dinner|   2|            8.49|Christy Cunningham|
|     10.34|1.66|  Male|    No|Sun|Dinner|   3|            3.45|    Douglas Tucker|
|     21.01| 3.5|  Male|    No|Sun|Dinner|   3|             7.0|    Travis Walters|
|     23.68|3.31|  Male|    No|Sun|Dinner|   2|           11.84|  Nathaniel Harris|
|     24.59|3.61|Female|    No|Sun|Dinner|   4|            6.15|      Tonya Carter|
|     25.29|4.71|  Male|    No|Sun|Dinner|   4|            6.32|        Erik Smith|
|      8.77| 2.0|  Male|    No|Sun|Dinner|   2|            4.38|Kristopher Johnson|
|     26.88|3.12|  Male|    No|Sun|Dinner|   4|            6.72|       Robert Buck|
|     15.04|1.96|  Male|    No|Sun|Dinner|   2|            7.52|   Joseph Mc

#### Creating a new column with name: tip_percentage 

In [5]:
#b.) Solution
import pyspark.sql.functions as F
tips_df = tips_df.withColumn('tip_percentage', F.expr("(tip/total_bill)*100"))

tips_df.show(10)

+----------+----+------+------+---+------+----+----------------+------------------+------------------+
|total_bill| tip|gender|smoker|day|  time|size|price_per_person|        Payer_Name|    tip_percentage|
+----------+----+------+------+---+------+----+----------------+------------------+------------------+
|     16.99|1.01|Female|    No|Sun|Dinner|   2|            8.49|Christy Cunningham|5.9446733372572105|
|     10.34|1.66|  Male|    No|Sun|Dinner|   3|            3.45|    Douglas Tucker|16.054158607350097|
|     21.01| 3.5|  Male|    No|Sun|Dinner|   3|             7.0|    Travis Walters|16.658733936220845|
|     23.68|3.31|  Male|    No|Sun|Dinner|   2|           11.84|  Nathaniel Harris| 13.97804054054054|
|     24.59|3.61|Female|    No|Sun|Dinner|   4|            6.15|      Tonya Carter|14.680764538430255|
|     25.29|4.71|  Male|    No|Sun|Dinner|   4|            6.32|        Erik Smith| 18.62396204033215|
|      8.77| 2.0|  Male|    No|Sun|Dinner|   2|            4.38|Kristophe

#### c.) registering the dataframe as temporary sql view- tips

In [6]:

tips_df.createOrReplaceTempView("tips")

tips_view = spark.sql("SELECT * FROM tips")
tips_view.show()

+----------+----+------+------+---+------+----+----------------+------------------+------------------+
|total_bill| tip|gender|smoker|day|  time|size|price_per_person|        Payer_Name|    tip_percentage|
+----------+----+------+------+---+------+----+----------------+------------------+------------------+
|     16.99|1.01|Female|    No|Sun|Dinner|   2|            8.49|Christy Cunningham|5.9446733372572105|
|     10.34|1.66|  Male|    No|Sun|Dinner|   3|            3.45|    Douglas Tucker|16.054158607350097|
|     21.01| 3.5|  Male|    No|Sun|Dinner|   3|             7.0|    Travis Walters|16.658733936220845|
|     23.68|3.31|  Male|    No|Sun|Dinner|   2|           11.84|  Nathaniel Harris| 13.97804054054054|
|     24.59|3.61|Female|    No|Sun|Dinner|   4|            6.15|      Tonya Carter|14.680764538430255|
|     25.29|4.71|  Male|    No|Sun|Dinner|   4|            6.32|        Erik Smith| 18.62396204033215|
|      8.77| 2.0|  Male|    No|Sun|Dinner|   2|            4.38|Kristophe

##### average total bill for each day

In [7]:
avg_total_bill = tips_view.groupBy('day').avg('total_bill')
avg_total_bill.show()

+----+------------------+
| day|   avg(total_bill)|
+----+------------------+
|Thur|17.682741935483865|
| Sun|21.410000000000004|
| Sat|20.441379310344825|
| Fri|17.151578947368417|
+----+------------------+



##### max tip given by gender

In [8]:
max_tip_by_gender = tips_view.groupBy('gender').max('tip')
max_tip_by_gender.show()

+------+--------+
|gender|max(tip)|
+------+--------+
|Female|     6.5|
|  Male|    10.0|
+------+--------+



##### display records where tip percentage is greater than 20%

In [10]:
from pyspark.sql.functions import col


# filtering the record using filter() 
#result_df = tips_view.filter("tip > 20")

tip_gt_20 = tips_view.where(col("tip_percentage") > 20)

tip_gt_20.show()

+----------+----+------+------+----+------+----+----------------+------------------+------------------+
|total_bill| tip|gender|smoker| day|  time|size|price_per_person|        Payer_Name|    tip_percentage|
+----------+----+------+------+----+------+----+----------------+------------------+------------------+
|      8.77| 2.0|  Male|    No| Sun|Dinner|   2|            4.38|Kristopher Johnson| 22.80501710376283|
|     14.78|3.23|  Male|    No| Sun|Dinner|   2|            7.39|     Jerome Abbott|21.853856562922868|
|     14.83|3.02|Female|    No| Sun|Dinner|   2|            7.42|     Vanessa Jones|20.364126770060686|
|     16.29|3.71|  Male|    No| Sun|Dinner|   3|            5.43|      John Pittman|22.774708410067525|
|     16.97| 3.5|Female|    No| Sun|Dinner|   3|            5.66|    Laura Martinez|20.624631703005306|
|     17.92|4.08|  Male|    No| Sat|Dinner|   2|            8.96|       Thomas Rice|22.767857142857142|
|     13.94|3.06|  Male|    No| Sun|Dinner|   2|            6.97

In [15]:
tip_gt_20.write.format("parquet").save(f"file:///home/talentum/shared/Q1/tips_data.parquet",mode = "overwrite")

Py4JJavaError: An error occurred while calling o204.save.
: org.apache.spark.SparkException: Job aborted.
	at org.apache.spark.sql.execution.datasources.FileFormatWriter$.write(FileFormatWriter.scala:198)
	at org.apache.spark.sql.execution.datasources.InsertIntoHadoopFsRelationCommand.run(InsertIntoHadoopFsRelationCommand.scala:159)
	at org.apache.spark.sql.execution.command.DataWritingCommandExec.sideEffectResult$lzycompute(commands.scala:104)
	at org.apache.spark.sql.execution.command.DataWritingCommandExec.sideEffectResult(commands.scala:102)
	at org.apache.spark.sql.execution.command.DataWritingCommandExec.doExecute(commands.scala:122)
	at org.apache.spark.sql.execution.SparkPlan$$anonfun$execute$1.apply(SparkPlan.scala:131)
	at org.apache.spark.sql.execution.SparkPlan$$anonfun$execute$1.apply(SparkPlan.scala:127)
	at org.apache.spark.sql.execution.SparkPlan$$anonfun$executeQuery$1.apply(SparkPlan.scala:155)
	at org.apache.spark.rdd.RDDOperationScope$.withScope(RDDOperationScope.sc

In [14]:
tip_gt_20.write.format("parquet").save(f"{filepath}/tips_data.parquet", mode="overwrite")

tip_gt_20.show()

Py4JJavaError: An error occurred while calling o157.save.
: org.apache.hadoop.fs.ParentNotDirectoryException: Parent path is not a directory: file:/home/talentum/shared/Q1/tips.csv
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:523)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:504)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:531)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:504)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:531)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:504)
	at org.apache.hadoop.fs.ChecksumFileSystem.mkdirs(ChecksumFileSystem.java:694)
	at org.apache.hadoop.mapreduce.lib.output.FileOutputCommitter.setupJob(FileOutputCommitter.java:313)
	at org.apache.spark.internal.io.HadoopMapReduceCommitProtocol.setupJob(HadoopMapReduceComm